In [1]:
import pickle
import numpy as np
import networkx as nx

In [2]:
# Get EDGE LIST
with open('loc-brightkite_edges.txt', 'r') as f:
    with open('edge_list.txt', 'w') as g:
        for line in f.readlines():
            x, y = line.split('\t')
            g.write(f"{int(x)} {int(y)}\n")

In [3]:
# GET ADJ_MATRIX
e = []
with open('loc-brightkite_edges.txt', 'r') as f:
    for line in f.readlines():
        x, y = line.split('\t')
        e.append((int(x), int(y)))

G = nx.Graph(e)

with open('nx_adj.pkl', 'wb') as f:
    pickle.dump(G, f)

In [2]:
import pandas as pd
df = pd.read_csv('loc-brightkite_totalCheckins.txt', sep='\t', header=None)

In [3]:
df.drop(labels=[1,2,3], axis=1, inplace=True)
df.rename(columns={0:"user", 4: "location"}, inplace=True)
df

,user,location
0,0,88c46bf20db295831bd2d1718ad7e6f5
1,0,7a0f88982aa015062b95e3b4843f9ca2
2,0,dd7cd3d264c2d063832db506fba8bf79
3,0,9848afcc62e500a01cf6fbf24b797732f8963683
4,0,2ef143e12038c870038df53e0478cefc
...,...,...
4747282,58222,9f6b83bca22411dd85460384f67fcdb0
4747283,58224,9f6b83bca22411dd85460384f67fcdb0
4747284,58225,9f6b83bca22411dd85460384f67fcdb0
4747285,58226,9f6b83bca22411dd85460384f67fcdb0


In [4]:
A = np.zeros((58228, 772967), dtype=bool)
print(A)

c = 0
mapper = dict() # location_str : column_index
for u, loc in df[['user', 'location']].values:
    if loc not in mapper:
        mapper[loc] = c
        c += 1
    A[u][mapper[loc]] = True

from scipy import sparse
A_sparse = sparse.csr_matrix(A)
sparse.save_npz("feature_matrix_full_sparse.npz", A_sparse)

[[False False False ... False False False]
 [False False False ... False False False]
 [False False False ... False False False]
 ...
 [False False False ... False False False]
 [False False False ... False False False]
 [False False False ... False False False]]


In [9]:
from scipy.stats import entropy
from tqdm import tqdm

def column_entropy_order(data):
    entropies = []
    for col_idx in tqdm(range(data.shape[1])):
        col = data[:, col_idx].astype(int)  # convert bools to 0/1
        counts = np.bincount(col, minlength=2)  # ensure both 0 and 1 are counted
        probs = counts / counts.sum()
        ent = entropy(probs, base=2)
        entropies.append((col_idx, ent))
    
    # Sort by entropy in descending order
    sorted_cols = sorted(entropies, key=lambda x: x[1], reverse=True)
    return [idx for idx, _ in sorted_cols]

cols = column_entropy_order(A)

100%|██████████████████████████████████| 772967/772967 [16:11<00:00, 795.81it/s]


In [10]:
A_50 = A[:, cols[:50]]

with open("feature_matrix_50_highest_entropy_dtype-bool.pkl", 'wb') as f:
    pickle.dump(A_50, f)

In [3]:
import pickle
import numpy as np

with open('feature_matrix_50_highest_entropy_dtype-bool.pkl', 'rb') as f:
    A_50 = pickle.load(f)

with open("CAGM_attribute.txt", 'w') as f:
    for i, row in enumerate(A_50):
        f.write(f"{int(i)} " + " ".join(str(int(e)) for e in row) + '\n')